# Uninformed Search Methods: DFS and BFS

**Author:** Phabel Antonio López Delgado, BSc.

This notebook explores two fundamental uninformed (blind) search algorithms:

- **Depth‑First Search (DFS)** – a recursive algorithm that explores as far as possible along each branch before backtracking.
- **Breadth‑First Search (BFS)** – an algorithm that explores all nodes at the present depth before moving on to nodes at the next depth level.

We apply DFS to a graph traversal problem and BFS to solve the classic 8‑puzzle (sliding tile puzzle).

**Key Techniques & Libraries:**
- `collections.deque` – for efficient queue operations (BFS).
- `numpy` – for matrix operations and comparisons.

**Objective:**
To understand the mechanics of uninformed search strategies, their implementation, and their application to real‑world problems such as graph navigation and puzzle solving.

In [1]:
import numpy as np
from collections import deque

# Set random seed for reproducibility
np.random.seed(42)

print("All imports successful.")

All imports successful.


## Depth‑First Search (DFS) on a Graph

We implement a recursive DFS that searches for a target node starting from a given origin. The algorithm returns the path from the origin to the target if found, or `None` otherwise.

In [2]:
def depth_first_search(graph, start=None, target=None, path=None):
    """
    Recursive depth-first search that finds a path from start to target.
    
    Parameters:
        graph (dict): adjacency list representation.
        start: starting node; if None, uses the first key.
        target: node to find.
        path (list): current path (used internally).
    
    Returns:
        list: path from start to target, or None if not found.
    """
    if start is None:
        start = next(iter(graph))
    
    if start not in graph:
        print("Invalid start node.")
        return None
    
    if path is None:
        path = [start]
    
    # Success: target found
    if start == target:
        print(f"Target {target} found in {len(path)-1} steps.")
        return path
    
    # Explore neighbors recursively
    for neighbor in graph[start]:
        if neighbor not in path:
            new_path = depth_first_search(graph, neighbor, target, path + [neighbor])
            if new_path:
                return new_path
    
    # Target not found in this branch
    return None

In [3]:
# Example graph (undirected)
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

# Test cases
test_cases = [
    {'start': 'Z', 'target': 'F'},
    {'start': None, 'target': 'Z'},
    {'start': 'A', 'target': 'A'},
    {'start': 'A', 'target': 'F'},
    {'start': 'E', 'target': 'F'},
    {'start': 'E', 'target': 'A'},
    {'start': 'A', 'target': 'C'},
    {'start': None, 'target': 'F'},
]

for case in test_cases:
    start = case.get('start')
    target = case['target']
    result = depth_first_search(graph, start, target)
    print(f"DFS from {start if start else 'first node'} to {target}: {result}\n")

Invalid start node.
DFS from Z to F: None

DFS from first node to Z: None

Target A found in 0 steps.
DFS from A to A: ['A']

Target F found in 3 steps.
DFS from A to F: ['A', 'B', 'E', 'F']

Target F found in 4 steps.
DFS from E to F: ['E', 'B', 'A', 'C', 'F']

Target A found in 2 steps.
DFS from E to A: ['E', 'B', 'A']

Target C found in 4 steps.
DFS from A to C: ['A', 'B', 'E', 'F', 'C']

Target F found in 3 steps.
DFS from first node to F: ['A', 'B', 'E', 'F']



## Solving the 8‑Puzzle using Breadth‑First Search (BFS)

The 8‑puzzle is a sliding puzzle with 8 numbered tiles and one empty space (0) on a 3×3 grid. We implement a `Puzzle` class that represents the board state and provides methods to generate legal moves (up, down, left, right). BFS explores all possible states level by level until it reaches the goal configuration.

We use a queue to manage the frontier and a set to track visited states to avoid cycles.

In [4]:
class Puzzle:
    """
    Represents the 8‑puzzle and provides BFS search to find the solution.
    """
    def __init__(self, initial_board):
        self.board = initial_board
        self.goal = [[1, 2, 3],
                     [4, 5, 6],
                     [7, 8, 0]]
        self.visited = set()
        self.moves = []   # stores the sequence of boards to reach the goal

    def _find_zero(self, board):
        """Return the (row, col) position of the empty tile (0)."""
        for i in range(3):
            for j in range(3):
                if board[i][j] == 0:
                    return i, j
        return None

    def _move_up(self, board):
        """Move the empty tile up if possible."""
        pos = self._find_zero(board)
        if pos is None or pos[0] == 0:
            return board
        i, j = pos
        new_board = [row[:] for row in board]
        new_board[i][j], new_board[i-1][j] = new_board[i-1][j], new_board[i][j]
        return new_board

    def _move_down(self, board):
        """Move the empty tile down if possible."""
        pos = self._find_zero(board)
        if pos is None or pos[0] == 2:
            return board
        i, j = pos
        new_board = [row[:] for row in board]
        new_board[i][j], new_board[i+1][j] = new_board[i+1][j], new_board[i][j]
        return new_board

    def _move_left(self, board):
        """Move the empty tile left if possible."""
        pos = self._find_zero(board)
        if pos is None or pos[1] == 0:
            return board
        i, j = pos
        new_board = [row[:] for row in board]
        new_board[i][j], new_board[i][j-1] = new_board[i][j-1], new_board[i][j]
        return new_board

    def _move_right(self, board):
        """Move the empty tile right if possible."""
        pos = self._find_zero(board)
        if pos is None or pos[1] == 2:
            return board
        i, j = pos
        new_board = [row[:] for row in board]
        new_board[i][j], new_board[i][j+1] = new_board[i][j+1], new_board[i][j]
        return new_board

    def _get_neighbors(self, board):
        """Generate all valid successor boards."""
        neighbors = []
        neighbors.append(self._move_up(board))
        neighbors.append(self._move_down(board))
        neighbors.append(self._move_left(board))
        neighbors.append(self._move_right(board))
        # Return only distinct boards (some moves may be invalid)
        unique = []
        for nb in neighbors:
            if nb != board and nb not in unique:
                unique.append(nb)
        return unique

    def _board_to_tuple(self, board):
        """Convert board to a tuple for hashing (visited set)."""
        return tuple(tuple(row) for row in board)

    def _print_solution(self, path):
        """Print the sequence of boards leading to the goal."""
        for step, board in enumerate(path):
            print(f"Step {step+1}:")
            for row in board:
                print(row)
            print()

    def solve_bfs(self):
        """
        Use breadth-first search to find the shortest path to the goal.
        Returns the path (list of boards) if found, else None.
        """
        start_tuple = self._board_to_tuple(self.board)
        if start_tuple == self._board_to_tuple(self.goal):
            print("Already solved!")
            self._print_solution([self.board])
            return [self.board]

        # Queue stores (board, path)
        queue = deque([(self.board, [self.board])])
        visited = {start_tuple}

        while queue:
            current, path = queue.popleft()
            neighbors = self._get_neighbors(current)
            for nb in neighbors:
                nb_tuple = self._board_to_tuple(nb)
                if nb_tuple not in visited:
                    visited.add(nb_tuple)
                    new_path = path + [nb]
                    if nb_tuple == self._board_to_tuple(self.goal):
                        print("Optimal solution found!")
                        self._print_solution(new_path)
                        return new_path
                    queue.append((nb, new_path))

        print("No solution found.")
        return None

In [5]:
# Initial configuration
initial_state = [
    [4, 1, 3],
    [7, 6, 0],
    [5, 2, 8]
]

puzzle = Puzzle(initial_state)
solution = puzzle.solve_bfs()

if solution:
    print("Number of steps:", len(solution) - 1)

Optimal solution found!
Step 1:
[4, 1, 3]
[7, 6, 0]
[5, 2, 8]

Step 2:
[4, 1, 3]
[7, 0, 6]
[5, 2, 8]

Step 3:
[4, 1, 3]
[7, 2, 6]
[5, 0, 8]

Step 4:
[4, 1, 3]
[7, 2, 6]
[0, 5, 8]

Step 5:
[4, 1, 3]
[0, 2, 6]
[7, 5, 8]

Step 6:
[0, 1, 3]
[4, 2, 6]
[7, 5, 8]

Step 7:
[1, 0, 3]
[4, 2, 6]
[7, 5, 8]

Step 8:
[1, 2, 3]
[4, 0, 6]
[7, 5, 8]

Step 9:
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]

Step 10:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

Number of steps: 9


## Summary and Next Steps

**Accomplished:**
- Implemented **Depth‑First Search (DFS)** recursively on an undirected graph, with path reconstruction and target validation.
- Tested DFS on multiple scenarios:
  - Invalid start node → handled gracefully.
  - Target not in graph → correctly returned `None`.
  - Target at root → found in 0 steps.
  - Target reachable from various start nodes → found valid paths.
- Implemented an **8‑puzzle solver** using **Breadth‑First Search (BFS)**, which guarantees the shortest path to the goal.
- The puzzle class includes all four directional moves (up, down, left, right) and tracks visited states to avoid cycles.
- Successfully found an optimal solution of **9 moves** for the given initial configuration.

**Key Results:**

| Search Method | Test Case | Outcome |
|---------------|-----------|---------|
| DFS | Graph traversal – valid start to target | Found valid paths for all reachable targets |
| DFS | Invalid start node | Returned `None` with error message |
| DFS | Target not in graph | Returned `None` |
| DFS | Target at root | Found in 0 steps |
| BFS | 8‑puzzle | Found optimal solution in 9 moves (10 boards) |

**Key Insights:**
- **DFS** is simple and memory‑efficient (only stores the current path), but it does not guarantee the shortest path and can get trapped in deep branches or infinite loops if cycles are not handled.
- **BFS** guarantees the shortest path but requires more memory (stores all nodes at the current frontier). It is ideal for problems where the optimal solution is required.
- The recursive implementation of DFS can be adapted to iterative using a stack for larger graphs, avoiding recursion depth limits.
- For the 8‑puzzle, BFS explores all states level by level; the branching factor is up to 4, making it feasible for this small problem (maximum 181,440 reachable states).
- Using a visited set prevents infinite loops and redundant exploration, significantly improving efficiency.
- The `deque` data structure provides efficient O(1) popleft operations for BFS.

**Suggested Next Steps:**
1. **Implement iterative DFS** – use a stack to avoid recursion depth limits for very large graphs.
2. **Add heuristic search** – implement A* for the 8‑puzzle using Manhattan distance or misplaced tiles heuristic, and compare performance with BFS.
3. **Visualise the search tree** – for both DFS and BFS, display the order of exploration to better understand their behaviour.
4. **Test BFS on random 8‑puzzle states** – measure the number of steps and states explored; identify unsolvable configurations using parity checks.
5. **Adapt the 8‑puzzle solver** – extend to larger puzzles (15‑puzzle, 24‑puzzle) using more advanced algorithms like IDA* with pattern databases.
6. **Compare DFS and BFS** – in terms of time and space complexity on larger graphs or more complex problems.
7. **Add bidirectional BFS** – implement a bidirectional search to reduce the search space and compare performance.

**Reflection:**
This notebook provides a hands‑on understanding of two fundamental uninformed search strategies. DFS demonstrates depth‑first exploration and recursion, while BFS illustrates breadth‑first exploration and its optimality guarantee for unweighted problems. The 8‑puzzle is a classic AI problem that highlights the importance of systematic search and state‑space representation. Together, these exercises build a solid foundation for more advanced search techniques such as informed (heuristic) search and adversarial search. The modular design of the puzzle class makes it easy to extend with new search methods or heuristics.